# The Decoupling of Phase and Structure (N4Entry)
### Computational Simulation of the $N_3 \to N_4$ Planetary Topological Snap and Phase Kinematics
**Author:** Tshuutheni Emvula (Independent Frontier Science Collaboration)  
**Associated Paper:** *The Decoupling of Phase and Structure: Differentiating Cognitive Synchronization from Biospheric Dissolution in the $N_3 \to N_4$ Topological Transition*


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Accelerating N4 Engine on: {device}")


## Part 1: Nested Dissipative NLKG Integration ($N_3 \to N_4$ Transition)


In [ ]:
def laplacian_3d_torch(field, dx=0.05):
    return (torch.roll(field, shifts=1, dims=0) + torch.roll(field, shifts=-1, dims=0) +
            torch.roll(field, shifts=1, dims=1) + torch.roll(field, shifts=-1, dims=1) +
            torch.roll(field, shifts=1, dims=2) + torch.roll(field, shifts=-1, dims=2) -
            6.0 * field) / (dx**2)

def step_dissipative_torch(phi_r, phi_i, phi_dot_r, phi_dot_i, dt=0.01, delta=0.015):
    """
    Integrates 3D complex scalar field with thermodynamic dissipation (Arrow of Time).
    """
    m_sq, g_attract, eta_stab = 1.0, -0.5, 0.01
    rho = phi_r**2 + phi_i**2
    
    force_r = laplacian_3d_torch(phi_r) - (m_sq * phi_r) - (g_attract * rho * phi_r) - (eta_stab * rho**2 * phi_r) - (delta * phi_dot_r)
    force_i = laplacian_3d_torch(phi_i) - (m_sq * phi_i) - (g_attract * rho * phi_i) - (eta_stab * rho**2 * phi_i) - (delta * phi_dot_i)
    
    phi_dot_r_new = phi_dot_r + force_r * dt
    phi_dot_i_new = phi_dot_i + force_i * dt
    phi_r_new = phi_r + phi_dot_r_new * dt
    phi_i_new = phi_i + phi_dot_i_new * dt
    
    theta_map = torch.atan2(torch.sqrt(phi_dot_r_new**2 + phi_dot_i_new**2), 
                            torch.sqrt(phi_r_new**2 + phi_i_new**2))
    
    mask = rho > 1e-3
    valid_thetas = torch.where(mask, theta_map, torch.zeros_like(theta_map))
    mask_sum = torch.clamp(torch.sum(mask), min=1.0)
    mean_theta = torch.sum(valid_thetas) / mask_sum
    variance_theta = torch.sum(mask * (valid_thetas - mean_theta)**2) / mask_sum
    
    return phi_r_new, phi_i_new, phi_dot_r_new, phi_dot_i_new, variance_theta.item()


## Part 2: $N_4$ Topological Catalogue Builder (Phase Kinematics of Consciousness)


In [ ]:
N, L, dt = 256, 20.0, 0.05
dx = L / N
grid = torch.linspace(-L/2, L/2, N, device=device)
X, Y = torch.meshgrid(grid, grid, indexing='ij')

pos_A, pos_B, radius = -2.5, 2.5, 1.5
envelope_A = torch.exp(-((X - pos_A)**2 + Y**2) / radius**2)
envelope_B = torch.exp(-((X - pos_B)**2 + Y**2) / radius**2)

def calc_friction(phi):
    grad_x, grad_y = torch.gradient(phi, spacing=dx)
    return grad_x**2 + grad_y**2

def laplacian_2d(field):
    return (torch.roll(field, shifts=1, dims=0) + torch.roll(field, shifts=-1, dims=0) +
            torch.roll(field, shifts=1, dims=1) + torch.roll(field, shifts=-1, dims=1) -
            4.0 * field) / (dx**2)

# State 1: Fear / Dissonance
phi_fear = envelope_A - envelope_B 
friction_fear = calc_friction(phi_fear)

# State 2: Forgiveness (High Dissipation Integration)
phi_forgive = phi_fear.clone()
phi_dot = torch.zeros_like(phi_forgive)
m_sq, g_attract, delta = 1.0, -0.8, 0.8

for step in range(150):
    lap = laplacian_2d(phi_forgive)
    force = lap - (m_sq * phi_forgive) - (g_attract * phi_forgive**3) - (delta * phi_dot)
    phi_dot = phi_dot + force * dt
    phi_forgive = phi_forgive + phi_dot * dt

friction_forgive = calc_friction(phi_forgive)

# State 3: Joy / Resonance
phi_joy = 1.4 * (envelope_A + envelope_B) 
friction_joy = calc_friction(phi_joy)

print("Topological Catalogue Tensors successfully constructed!")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(friction_fear.cpu().numpy(), cmap='hot')
axes[0].set_title("State 1: Fear (Dissonance / Friction)")
axes[1].imshow(friction_forgive.cpu().numpy(), cmap='plasma')
axes[1].set_title("State 2: Forgiveness (Dissipative Annealing)")
axes[2].imshow(friction_joy.cpu().numpy(), cmap='viridis')
axes[2].set_title("State 3: Joy (Harmonic Resonance)")
plt.tight_layout()
plt.show()
